In [1]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

In [2]:
ROOT          = "./datasets/1_NCA"
SEQ_LEN       = 50          # look-back window (time-steps)
BATCH_SIZE    = 64
EPOCHS        = 20
LR            = 1e-3
HIDDEN_LSTM   = 128
CNN_CHANNELS  = 64
DROPOUT       = 0.3
LABEL_MODE    = "both"      # "cc" | "norm_qchg" | "both"
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

FEATURES = ["Ecell/V", "<I>/mA", "Q discharge/mA.h", "Q charge/mA.h"]

print(f"Device : {DEVICE}")
print(f"Label  : {LABEL_MODE}\n")

Device : cuda
Label  : both



# LOAD DATASET AND PROCESS.

In [3]:
# 1.  LOAD & CLEAN DATA
def load_all(root: str) -> pd.DataFrame:
    frames = []
    for path in sorted(glob.glob(os.path.join(root, "*.csv"))):
        df = pd.read_csv(path)
        df.columns = df.columns.str.strip()
        df["source_file"] = os.path.basename(path)
        frames.append(df)
    data = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(frames)} files  →  {len(data):,} rows total")
    return data

raw = load_all(ROOT)

# Keep only charge half-cycles (current > 0, Q_chg increasing)
raw = raw[raw["<I>/mA"] > 0].copy()
raw = raw.dropna(subset=FEATURES + ["cycle number"])
raw = raw.reset_index(drop=True)
print(f"After charge-only filter: {len(raw):,} rows\n")

Loaded 20 files  →  1,862,233 rows total
After charge-only filter: 1,201,187 rows



In [4]:
# 2.  LABEL GENERATION
def coulomb_counting_soc(group: pd.DataFrame) -> pd.Series:
    """
    SoC_CC = cumulative charge (A·s) / total charge of the cycle.
    Δt in seconds, I in mA → convert mA·s → mA·h via /3600.
    """
    t  = group["time/s"].values
    I  = group["<I>/mA"].values
    dt = np.diff(t, prepend=t[0])
    dQ = I * dt / 3600.0          # incremental charge in mA·h
    Q_cum  = np.cumsum(dQ)
    Q_total = Q_cum[-1]
    if Q_total <= 0:
        return pd.Series(np.zeros(len(group)), index=group.index)
    soc = np.clip(Q_cum / Q_total, 0.0, 1.0)
    return pd.Series(soc, index=group.index)


def norm_qchg_soc(group: pd.DataFrame) -> pd.Series:
    """
    SoC_norm = Q_chg / Q_chg_max   (already accumulated in dataset)
    """
    q   = group["Q charge/mA.h"].values
    q_max = q.max()
    if q_max <= 0:
        return pd.Series(np.zeros(len(group)), index=group.index)
    soc = np.clip(q / q_max, 0.0, 1.0)
    return pd.Series(soc, index=group.index)

In [5]:
# Group by file + cycle for independent integration
grp_keys = ["source_file", "cycle number"]
raw["SoC_CC"]       = raw.groupby(grp_keys, group_keys=False).apply(coulomb_counting_soc)
raw["SoC_norm_Qchg"]= raw.groupby(grp_keys, group_keys=False).apply(norm_qchg_soc)

print("Label stats (SoC_CC):")
print(raw["SoC_CC"].describe().round(4))
print("\nLabel stats (SoC_norm_Qchg):")
print(raw["SoC_norm_Qchg"].describe().round(4))

Label stats (SoC_CC):
count    1.201187e+06
mean     6.768000e-01
std      3.504000e-01
min      0.000000e+00
25%      3.999000e-01
50%      8.533000e-01
75%      9.563000e-01
max      1.000000e+00
Name: SoC_CC, dtype: float64

Label stats (SoC_norm_Qchg):
count    1.201187e+06
mean     6.768000e-01
std      3.504000e-01
min      0.000000e+00
25%      3.999000e-01
50%      8.533000e-01
75%      9.563000e-01
max      1.000000e+00
Name: SoC_norm_Qchg, dtype: float64


In [6]:
# 3.  FEATURE SCALING
feat_scaler = MinMaxScaler()
raw[FEATURES] = feat_scaler.fit_transform(raw[FEATURES])

In [7]:
# 4.  SLIDING-WINDOW SEQUENCE BUILDER
def build_sequences(df: pd.DataFrame, seq_len: int, label_col: str):
    """
    Build (X, y) arrays using a sliding window within each cycle.
    X shape: (N, seq_len, n_features)
    y shape: (N,)
    """
    X_list, y_list = [], []
    groups = df.groupby(["source_file", "cycle number"])
    for _, g in groups:
        g = g.reset_index(drop=True)
        feats = g[FEATURES].values          # (T, F)
        labels = g[label_col].values        # (T,)
        T = len(g)
        if T <= seq_len:
            continue
        for i in range(T - seq_len):
            X_list.append(feats[i : i + seq_len])
            y_list.append(labels[i + seq_len])
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

print("\nBuilding sequences …")
X_cc,   y_cc   = build_sequences(raw, SEQ_LEN, "SoC_CC")
X_nq,   y_nq   = build_sequences(raw, SEQ_LEN, "SoC_norm_Qchg")
print(f"  CC  sequences: X={X_cc.shape}, y={y_cc.shape}")
print(f"  NQ  sequences: X={X_nq.shape}, y={y_nq.shape}")


Building sequences …
  CC  sequences: X=(1094087, 50, 4), y=(1094087,)
  NQ  sequences: X=(1094087, 50, 4), y=(1094087,)


In [8]:
# 5.  DATASET & DATALOADER
class BatteryDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_loaders(X, y):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.15,
                                               random_state=42, shuffle=True)
    X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.15,
                                               random_state=42, shuffle=True)
    tr = DataLoader(BatteryDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    va = DataLoader(BatteryDataset(X_va, y_va), batch_size=BATCH_SIZE)
    te = DataLoader(BatteryDataset(X_te, y_te), batch_size=BATCH_SIZE)
    return tr, va, te

In [9]:
# 6.  CNN-LSTM MODEL
class CNN_LSTM(nn.Module):
    """
    Architecture:
      Input  : (B, T, F)
      CNN    : 1-D temporal convolutions on the feature dimension
      LSTM   : captures long-range temporal dependencies
      Head   : FC layers → scalar SoC in [0, 1]
    """
    def __init__(self, n_features: int, seq_len: int,
                 cnn_channels: int = 64, hidden_lstm: int = 128,
                 dropout: float = 0.3):
        super().__init__()

        # ── CNN branch ──────────────────────────────────────
        # Input: (B, F, T)  (conv1d expects channels-first)
        self.cnn = nn.Sequential(
            nn.Conv1d(n_features, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # ── LSTM branch ─────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size  = cnn_channels,
            hidden_size = hidden_lstm,
            num_layers  = 2,
            batch_first = True,
            dropout     = dropout,
            bidirectional = False,
        )

        # ── Regression head ──────────────────────────────────
        self.head = nn.Sequential(
            nn.Linear(hidden_lstm, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid(),            # SoC ∈ [0, 1]
        )

    def forward(self, x):
        # x: (B, T, F)
        cnn_in  = x.permute(0, 2, 1)          # → (B, F, T)
        cnn_out = self.cnn(cnn_in)             # → (B, C, T)
        lstm_in = cnn_out.permute(0, 2, 1)    # → (B, T, C)
        lstm_out, _ = self.lstm(lstm_in)       # → (B, T, H)
        last = lstm_out[:, -1, :]              # last time-step
        return self.head(last)

In [10]:
# 7.  TRAIN / EVAL HELPERS
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X_b)
        loss = criterion(pred, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(X_b)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    preds, trues = [], []
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        pred = model(X_b)
        total_loss += criterion(pred, y_b).item() * len(X_b)
        preds.append(pred.cpu().numpy())
        trues.append(y_b.cpu().numpy())
    preds = np.concatenate(preds).flatten()
    trues = np.concatenate(trues).flatten()
    return total_loss / len(loader.dataset), preds, trues


In [11]:
def run_experiment(label_name: str, X: np.ndarray, y: np.ndarray):
    print(f"\n{'='*60}")
    print(f"  EXPERIMENT: {label_name}")
    print(f"{'='*60}")

    tr_loader, va_loader, te_loader = make_loaders(X, y)

    model     = CNN_LSTM(len(FEATURES), SEQ_LEN, CNN_CHANNELS, HIDDEN_LSTM, DROPOUT).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5,
                                                            factor=0.5, verbose=True)

    print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

    best_val   = float("inf")
    best_state = None
    history    = {"train": [], "val": []}

    for epoch in range(1, EPOCHS + 1):
        tr_loss = train_one_epoch(model, tr_loader, optimizer, criterion)
        va_loss, _, _ = evaluate(model, va_loader, criterion)
        scheduler.step(va_loss)

        history["train"].append(tr_loss)
        history["val"].append(va_loss)

        if va_loss < best_val:
            best_val   = va_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{EPOCHS}  │  train MSE={tr_loss:.5f}  │  val MSE={va_loss:.5f}")

    # ── Test with best checkpoint ──
    model.load_state_dict(best_state)
    _, preds, trues = evaluate(model, te_loader, criterion)

    mae  = mean_absolute_error(trues, preds)
    rmse = np.sqrt(mean_squared_error(trues, preds))
    r2   = r2_score(trues, preds)

    print(f"\n  Test Results  │  MAE={mae:.4f}  │  RMSE={rmse:.4f}  │  R²={r2:.4f}")

    # ── Save model ──
    safe_name = label_name.replace(" ", "_").replace("/", "_")
    ckpt_path = f"cnn_lstm_soc_{safe_name}.pt"
    torch.save({"model_state": best_state, "label": label_name,
                "seq_len": SEQ_LEN, "features": FEATURES}, ckpt_path)
    print(f"  Checkpoint saved → {ckpt_path}")

    return model, history, preds, trues, {"MAE": mae, "RMSE": rmse, "R2": r2}

# 8.  RUN EXPERIMENTS
results = {}

if LABEL_MODE in ("cc", "both"):
    m_cc, h_cc, p_cc, t_cc, met_cc = run_experiment("Coulomb Counting (CC)", X_cc, y_cc)
    results["CC"] = (h_cc, p_cc, t_cc, met_cc)

if LABEL_MODE in ("norm_qchg", "both"):
    m_nq, h_nq, p_nq, t_nq, met_nq = run_experiment("Normalized Q_chg", X_nq, y_nq)
    results["NQ"] = (h_nq, p_nq, t_nq, met_nq)


  EXPERIMENT: Coulomb Counting (CC)

Model parameters: 253,185
  Epoch   1/20  │  train MSE=0.00084  │  val MSE=0.00017
  Epoch  10/20  │  train MSE=0.00066  │  val MSE=0.00011


KeyboardInterrupt: 

In [ ]:
# 9.  PLOTTING
n_experiments = len(results)
fig, axes = plt.subplots(n_experiments, 2,
                         figsize=(14, 5 * n_experiments), squeeze=False)
fig.suptitle("CNN-LSTM SoC Prediction — NCA Battery", fontsize=16, fontweight="bold")

titles = {"CC": "Coulomb Counting", "NQ": "Normalized Q_chg"}
colors = {"CC": ("#2196F3", "#F44336"), "NQ": ("#4CAF50", "#FF9800")}

for row, (key, (hist, preds, trues, metrics)) in enumerate(results.items()):
    ax_loss = axes[row][0]
    ax_pred = axes[row][1]

    # Loss curve
    c_tr, c_va = colors[key]
    ax_loss.plot(hist["train"], label="Train MSE", color=c_tr, lw=1.5)
    ax_loss.plot(hist["val"],   label="Val MSE",   color=c_va, lw=1.5, linestyle="--")
    ax_loss.set_title(f"{titles[key]} — Training Curve")
    ax_loss.set_xlabel("Epoch"); ax_loss.set_ylabel("MSE Loss")
    ax_loss.legend(); ax_loss.grid(alpha=0.3)

    # Prediction scatter (first 1000 for clarity)
    n = min(1000, len(trues))
    ax_pred.scatter(trues[:n], preds[:n], alpha=0.4, s=8, color=c_tr)
    lims = [0, 1]
    ax_pred.plot(lims, lims, "k--", lw=1.2, label="Perfect")
    ax_pred.set_xlim(lims); ax_pred.set_ylim(lims)
    ax_pred.set_title(
        f"{titles[key]} — Parity Plot\n"
        f"MAE={metrics['MAE']:.4f} | RMSE={metrics['RMSE']:.4f} | R²={metrics['R2']:.4f}"
    )
    ax_pred.set_xlabel("True SoC"); ax_pred.set_ylabel("Predicted SoC")
    ax_pred.legend(); ax_pred.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("cnn_lstm_soc_results.png", dpi=150)
print("\nPlot saved → cnn_lstm_soc_results.png")
plt.show()

# 10.  SUMMARY TABLE
print("\n" + "─" * 45)
print(f"{'Method':<25} {'MAE':>7} {'RMSE':>8} {'R²':>7}")
print("─" * 45)
for key, (_, _, _, m) in results.items():
    print(f"{titles[key]:<25} {m['MAE']:>7.4f} {m['RMSE']:>8.4f} {m['R2']:>7.4f}")
print("─" * 45)